# Fine-tune `abdullahg7/cardd-yolov8s` on VehiDE

In [2]:
!pip install -q ultralytics huggingface_hub

import json, shutil, time
from pathlib import Path

import torch

assert torch.cuda.is_available(), "Enable GPU in Settings → Accelerator before running."
print("GPU:", torch.cuda.get_device_name(0))

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 30.6 MB/s eta 0:00:0000:01
GPU: Tesla T4


## 1. Mount the converted segmentation dataset

In [3]:
!ls /kaggle/input/

SEG_ROOT = Path("/kaggle/input/datasets/m4rcuseryx/vehide-segmentation-dataset")
SEG_IMAGES_LABELS_ROOT = SEG_ROOT / "vehide_seg"
SEG_DATA_YAML_SOURCE = SEG_ROOT / "damage-seg.yaml"

assert SEG_ROOT.exists(), "Segmentation dataset not found at this path — check /kaggle/input/ above and update SEG_ROOT."
assert SEG_IMAGES_LABELS_ROOT.exists(), f"Expected images/labels under {SEG_IMAGES_LABELS_ROOT}, not found."
assert SEG_DATA_YAML_SOURCE.exists(), f"Expected damage-seg.yaml at {SEG_DATA_YAML_SOURCE}, not found."

print(open(SEG_DATA_YAML_SOURCE).read())

CLASS_NAMES = ["dent", "scratch", "crack", "broken_lamp", "shattered_glass", "flat_tyre"]

datasets
names:
- dent
- scratch
- crack
- broken_lamp
- shattered_glass
- flat_tyre
nc: 6
path: /kaggle/working/vehide_seg
test: images/test
train: images/train
val: images/val



In [4]:
import yaml

with open(SEG_DATA_YAML_SOURCE) as f:
    cfg = yaml.safe_load(f)
cfg["path"] = str(SEG_IMAGES_LABELS_ROOT)

SEG_DATA_YAML = "/kaggle/working/damage-seg.yaml"
with open(SEG_DATA_YAML, "w") as f:
    yaml.safe_dump(cfg, f)
print(open(SEG_DATA_YAML).read())

names:
- dent
- scratch
- crack
- broken_lamp
- shattered_glass
- flat_tyre
nc: 6
path: /kaggle/input/datasets/m4rcuseryx/vehide-segmentation-dataset/vehide_seg
test: images/test
train: images/train
val: images/val



## 2. Download the pretrained CarDD checkpoint

In [5]:
from huggingface_hub import hf_hub_download
from ultralytics import YOLO

cardd_weights = hf_hub_download(repo_id="abdullahg7/cardd-yolov8s", filename="v2.0/best.pt")
print("Downloaded:", cardd_weights)

probe = YOLO(cardd_weights)
print("\nPretrained CarDD model's own class list (for reference — not necessarily")
print("the same order as this project's 6 classes; the head is reinitialised below")
print("since we are fine-tuning on our own class set):")
print(probe.names)
print("\nTask:", probe.task)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


v2.0/best.pt:   0%|          | 0.00/23.9M [00:00<?, ?B/s]

Downloaded: /root/.cache/huggingface/hub/models--abdullahg7--cardd-yolov8s/snapshots/9ee0339097a4c01cc70d5a61bc397c5026032260/v2.0/best.pt

Pretrained CarDD model's own class list (for reference — not necessarily
the same order as this project's 6 classes; the head is reinitialised below
since we are fine-tuning on our own class set):
{0: 'dent', 1: 'scratch', 2: 'crack', 3: 'glass_shatter', 4: 'lamp_broken', 5: 'tire_flat'}

Task: segment


## 3. Fine-tune, with multi-session checkpoint relay

In [6]:
import os
import subprocess
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
os.environ["KAGGLE_USERNAME"] = secrets.get_secret("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = secrets.get_secret("KAGGLE_KEY")

auth_check = subprocess.run(
    ["kaggle", "datasets", "list", "-s", "zzz_auth_check_zzz", "-p", "1"],
    capture_output=True, text=True,
)
if auth_check.returncode != 0:
    print("STDOUT:", auth_check.stdout)
    print("STDERR:", auth_check.stderr)
    raise RuntimeError(
        "Kaggle API authentication failed. Checkpoint backups will NOT work "
        "until this is fixed. Check:\n"
        "  1. Add-ons -> Secrets -> confirm KAGGLE_USERNAME and KAGGLE_KEY are "
        "both added AND toggled ON for this notebook (Kaggle requires "
        "per-notebook attachment, not just creating the secret once).\n"
        "  2. The values match your actual kaggle.json exactly (re-download a "
        "fresh token from kaggle.com/settings -> API -> Create New Token if unsure).\n"
        "  3. Consider upgrading the kaggle package: !pip install -q -U kaggle"
    )
print("Kaggle API authentication OK.")

RUN_NAME = "cardd_yolov8s_seg_finetune"

SESSION_EPOCH_BUDGET = 999   # effectively "never stop early"
BACKUP_EVERY_N_EPOCHS = 5
BACKUP_SLUG = "cardd-seg-finetune-checkpoint-backup"
BACKUP_DATASET_ID = f"{os.environ['KAGGLE_USERNAME']}/{BACKUP_SLUG}"

BACKUP_STAGE = Path("/kaggle/working/checkpoint_backup_seg")
BACKUP_STAGE.mkdir(parents=True, exist_ok=True)

SEG_TRAIN_ARGS = dict(
    data=SEG_DATA_YAML,
    epochs=40,
    imgsz=1280,
    batch=4,
    optimizer="AdamW",
    lr0=0.0005,
    lrf=0.001,
    cos_lr=True,
    weight_decay=0.0005,
    warmup_epochs=3,
    cls=0.5,
    patience=15,
    seed=42,
    deterministic=True,
    plots=True,
    project="/kaggle/working/runs/cardd_seg",
)
print(json.dumps(SEG_TRAIN_ARGS, indent=2))

Kaggle API authentication OK.
{
  "data": "/kaggle/working/damage-seg.yaml",
  "epochs": 40,
  "imgsz": 1280,
  "batch": 4,
  "optimizer": "AdamW",
  "lr0": 0.0005,
  "lrf": 0.001,
  "cos_lr": true,
  "weight_decay": 0.0005,
  "warmup_epochs": 3,
  "cls": 0.5,
  "patience": 15,
  "seed": 42,
  "deterministic": true,
  "plots": true,
  "project": "/kaggle/working/runs/cardd_seg"
}


In [7]:
meta_check = subprocess.run(
    ["kaggle", "datasets", "metadata", BACKUP_DATASET_ID, "-p", "/tmp/meta_check_seg"],
    capture_output=True, text=True,
)
check = meta_check.returncode
resume_from = None

if check == 0:
    print(f"Found existing backup: {BACKUP_DATASET_ID} — downloading...")
    os.makedirs("/kaggle/working/recovered_seg", exist_ok=True)
    dl = subprocess.run(
        ["kaggle", "datasets", "download", BACKUP_DATASET_ID,
         "-p", "/kaggle/working/recovered_seg", "--unzip", "-q"],
        capture_output=True, text=True,
    )
    if dl.returncode != 0:
        print(dl.stdout, dl.stderr)
        raise RuntimeError("Backup dataset metadata was found, but downloading it failed. "
                          "Check the output above before proceeding.")
    recovered_pt = list(Path("/kaggle/working/recovered_seg").rglob("last.pt"))
    if recovered_pt:
        resume_from = recovered_pt[0]
        epoch_marker = Path("/kaggle/working/recovered_seg/epoch.txt")
        print(f"Will resume from epoch {epoch_marker.read_text().strip() if epoch_marker.exists() else '?'}")
else:
    print(f"No existing backup dataset ({BACKUP_DATASET_ID}) — session 1, "
          f"starting fresh from the CarDD checkpoint.")
    print(f"(kaggle CLI returned: {meta_check.stderr.strip()[:200]})")

Found existing backup: m4rcuseryx/cardd-seg-finetune-checkpoint-backup — downloading...
Will resume from epoch 15


In [8]:
_session_start_epoch = {"value": None}
_dataset_exists = {"value": check == 0}
_backup_failures = []

def _push_backup(trainer, completed_epoch):
    last_pt = trainer.save_dir / "weights" / "last.pt"
    if not last_pt.exists():
        return
    shutil.copy(last_pt, BACKUP_STAGE / "last.pt")
    for extra in ("results.csv", "args.yaml"):
        p = trainer.save_dir / extra
        if p.exists():
            shutil.copy(p, BACKUP_STAGE / extra)
    (BACKUP_STAGE / "epoch.txt").write_text(str(completed_epoch))
    (BACKUP_STAGE / "dataset-metadata.json").write_text(json.dumps({
        "title": "CarDD YOLOv8s-seg fine-tune checkpoint backup",
        "id": BACKUP_DATASET_ID,
        "licenses": [{"name": "CC0-1.0"}],
    }))

    if not _dataset_exists["value"]:
        result = subprocess.run(
            ["kaggle", "datasets", "create", "-p", str(BACKUP_STAGE), "--dir-mode", "zip", "-q"],
            capture_output=True, text=True,
        )
        if result.returncode == 0:
            _dataset_exists["value"] = True
    else:
        result = subprocess.run(
            ["kaggle", "datasets", "version", "-p", str(BACKUP_STAGE),
             "-m", f"epoch {completed_epoch}", "--dir-mode", "zip", "-q"],
            capture_output=True, text=True,
        )

    if result.returncode == 0:
        print(f"Backed up checkpoint at epoch {completed_epoch} -> {BACKUP_DATASET_ID}")
    else:
        msg = f"BACKUP FAILED at epoch {completed_epoch}: {result.stderr.strip()[:300]}"
        print(f"\n{'='*70}\n{msg}\n{'='*70}\n")
        _backup_failures.append((completed_epoch, result.stderr.strip()))


def relay_callback(trainer):
    completed = trainer.epoch + 1
    if _session_start_epoch["value"] is None:
        _session_start_epoch["value"] = completed - 1
    epochs_this_session = completed - _session_start_epoch["value"]
    is_budget_stop = epochs_this_session >= SESSION_EPOCH_BUDGET
    is_periodic_backup = completed % BACKUP_EVERY_N_EPOCHS == 0

    if is_budget_stop or is_periodic_backup:
        _push_backup(trainer, completed)
    if is_budget_stop:
        print(f"\nSession budget of {SESSION_EPOCH_BUDGET} epochs reached "
              f"(total completed: {completed}/{trainer.epochs}). Stopping gracefully.")
        trainer.stop = True


t0 = time.time()
if resume_from is not None:
    model_seg = YOLO(str(resume_from))
    model_seg.add_callback("on_train_epoch_end", relay_callback)
    results_seg = model_seg.train(resume=True)
else:
    model_seg = YOLO(cardd_weights)
    model_seg.add_callback("on_train_epoch_end", relay_callback)
    results_seg = model_seg.train(name=RUN_NAME, **SEG_TRAIN_ARGS)
wall = time.time() - t0
print(f"\nThis session: {wall/60:.1f} min")

if _backup_failures:
    print(f"\nWARNING: {len(_backup_failures)} backup attempt(s) failed this session. "
          f"If the session dies before you fix this, progress since the last "
          f"SUCCESSFUL backup will be lost. Failed epochs: "
          f"{[e for e, _ in _backup_failures]}")
else:
    print("\nAll backup attempts this session succeeded.")

Ultralytics 8.4.108 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/kaggle/working/damage-seg.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1280, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0005, lrf=0.001, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/kaggle/working/recovered_seg/last.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=c

## 4. Evaluate

In [9]:
m = model_seg.val(data=SEG_DATA_YAML, split="test", imgsz=1280)
print("Box    mAP50:", float(m.box.map50), " mAP50-95:", float(m.box.map))
print("Mask   mAP50:", float(m.seg.map50), " mAP50-95:", float(m.seg.map))

import pandas as pd
rows = []
for idx, ci in enumerate(m.box.ap_class_index):
    rows.append({
        "class": CLASS_NAMES[int(ci)],
        "box_mAP50": float(m.box.ap50[idx]),
        "mask_mAP50": float(m.seg.ap50[idx]),
    })
pd.DataFrame(rows).sort_values("mask_mAP50", ascending=False)

Ultralytics 8.4.108 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
YOLOv8s-seg summary (fused): 86 layers, 11,781,922 parameters, 0 gradients, 39.9 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 0.0±0.0 ms, read: 10.1±7.1 MB/s, size: 249.1 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips/
val: Scanning /kaggle/input/datasets/m4rcuseryx/vehide-segmentation-dataset/vehide_seg/labels/test... 2047 images, 140 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 2047/2047 180.0it/s 11.4s0.1s
WARNING ⚠️ val: Cache directory /kaggle/input/datasets/m4rcuseryx/vehide-segmentation-dataset/vehide_seg/labels is not writable, cache not saved.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 128/128 1.2s/it 2:291.1ss
                   all       2047       4846      0.466      0.402      0.37

,class,box_mAP50,mask_mAP50
4,shattered_glass,0.702031,0.675127
5,flat_tyre,0.451360,0.470781
3,broken_lamp,0.405097,0.317771
0,dent,0.234416,0.226248
1,scratch,0.263079,0.202366
2,crack,0.210474,0.197781


## Continuation

In [11]:
CONTINUE_RUN_NAME = "cardd_yolov8s_seg_continued"

CONTINUE_SESSION_EPOCH_BUDGET = 999
CONTINUE_BACKUP_EVERY_N_EPOCHS = 5
CONTINUE_BACKUP_SLUG = "cardd-seg-continuation-checkpoint-backup"
CONTINUE_BACKUP_DATASET_ID = f"{os.environ['KAGGLE_USERNAME']}/{CONTINUE_BACKUP_SLUG}"

CONTINUE_BACKUP_STAGE = Path("/kaggle/working/checkpoint_backup_seg_continue")
CONTINUE_BACKUP_STAGE.mkdir(parents=True, exist_ok=True)

BEST_PT_FROM_ORIGINAL_RUN = "/kaggle/working/runs/cardd_seg/cardd_yolov8s_seg_finetune-2/weights/best.pt"

CONTINUE_ARGS = dict(
    data=SEG_DATA_YAML,
    epochs=20,
    imgsz=1280,
    batch=4,
    optimizer="AdamW",
    lr0=0.0002,          
    lrf=0.001,
    cos_lr=True,
    weight_decay=0.0005,
    warmup_epochs=1, 
    cls=0.5,
    patience=10,
    close_mosaic=5,
    copy_paste=0.3,
    hsv_v=0.5,
    hsv_s=0.8,
    degrees=10.0,
    shear=5.0,
    scale=0.6,
    seed=42,
    deterministic=True,
    plots=True,
    project="/kaggle/working/runs/cardd_seg",
)
print(json.dumps(CONTINUE_ARGS, indent=2))


{
  "data": "/kaggle/working/damage-seg.yaml",
  "epochs": 20,
  "imgsz": 1280,
  "batch": 4,
  "optimizer": "AdamW",
  "lr0": 0.0002,
  "lrf": 0.001,
  "cos_lr": true,
  "weight_decay": 0.0005,
  "warmup_epochs": 1,
  "cls": 0.5,
  "patience": 10,
  "close_mosaic": 5,
  "copy_paste": 0.3,
  "hsv_v": 0.5,
  "hsv_s": 0.8,
  "degrees": 10.0,
  "shear": 5.0,
  "scale": 0.6,
  "seed": 42,
  "deterministic": true,
  "plots": true,
  "project": "/kaggle/working/runs/cardd_seg"
}


In [12]:
continue_meta_check = subprocess.run(
    ["kaggle", "datasets", "metadata", CONTINUE_BACKUP_DATASET_ID, "-p", "/tmp/meta_check_continue"],
    capture_output=True, text=True,
)
continue_check = continue_meta_check.returncode
continue_resume_from = None

if continue_check == 0:
    print(f"Found existing continuation backup: {CONTINUE_BACKUP_DATASET_ID} — downloading...")
    os.makedirs("/kaggle/working/recovered_continue", exist_ok=True)
    dl = subprocess.run(
        ["kaggle", "datasets", "download", CONTINUE_BACKUP_DATASET_ID,
         "-p", "/kaggle/working/recovered_continue", "--unzip", "-q"],
        capture_output=True, text=True,
    )
    if dl.returncode != 0:
        print(dl.stdout, dl.stderr)
        raise RuntimeError("Continuation backup metadata was found, but downloading it failed.")
    recovered_pt = list(Path("/kaggle/working/recovered_continue").rglob("last.pt"))
    if recovered_pt:
        continue_resume_from = recovered_pt[0]
        epoch_marker = Path("/kaggle/working/recovered_continue/epoch.txt")
        print(f"Will resume the CONTINUATION run from epoch "
              f"{epoch_marker.read_text().strip() if epoch_marker.exists() else '?'}")
else:
    print(f"No existing continuation backup — this is the first session of the "
          f"continuation phase. Starting fresh from {BEST_PT_FROM_ORIGINAL_RUN}")
    print(f"(kaggle CLI returned: {continue_meta_check.stderr.strip()[:200]})")


No existing continuation backup — this is the first session of the continuation phase. Starting fresh from /kaggle/working/runs/cardd_seg/cardd_yolov8s_seg_finetune-2/weights/best.pt
(kaggle CLI returned: )


In [14]:
_continue_session_start_epoch = {"value": None}
_continue_dataset_exists = {"value": continue_check == 0}
_continue_backup_failures = []


def _push_continue_backup(trainer, completed_epoch):
    last_pt = trainer.save_dir / "weights" / "last.pt"
    if not last_pt.exists():
        return
    shutil.copy(last_pt, CONTINUE_BACKUP_STAGE / "last.pt")
    for extra in ("results.csv", "args.yaml"):
        p = trainer.save_dir / extra
        if p.exists():
            shutil.copy(p, CONTINUE_BACKUP_STAGE / extra)
    (CONTINUE_BACKUP_STAGE / "epoch.txt").write_text(str(completed_epoch))
    (CONTINUE_BACKUP_STAGE / "dataset-metadata.json").write_text(json.dumps({
        "title": "CarDD YOLOv8s-seg continuation checkpoint backup",
        "id": CONTINUE_BACKUP_DATASET_ID,
        "licenses": [{"name": "CC0-1.0"}],
    }))

    if not _continue_dataset_exists["value"]:
        result = subprocess.run(
            ["kaggle", "datasets", "create", "-p", str(CONTINUE_BACKUP_STAGE), "--dir-mode", "zip", "-q"],
            capture_output=True, text=True,
        )
        if result.returncode == 0:
            _continue_dataset_exists["value"] = True
    else:
        result = subprocess.run(
            ["kaggle", "datasets", "version", "-p", str(CONTINUE_BACKUP_STAGE),
             "-m", f"epoch {completed_epoch}", "--dir-mode", "zip", "-q"],
            capture_output=True, text=True,
        )

    if result.returncode == 0:
        print(f"Backed up continuation checkpoint at epoch {completed_epoch} -> {CONTINUE_BACKUP_DATASET_ID}")
    else:
        msg = f"BACKUP FAILED at epoch {completed_epoch}: {result.stderr.strip()[:300]}"
        print(f"\n{'='*70}\n{msg}\n{'='*70}\n")
        _continue_backup_failures.append((completed_epoch, result.stderr.strip()))


def continue_relay_callback(trainer):
    completed = trainer.epoch + 1
    if _continue_session_start_epoch["value"] is None:
        _continue_session_start_epoch["value"] = completed - 1
    epochs_this_session = completed - _continue_session_start_epoch["value"]
    is_budget_stop = epochs_this_session >= CONTINUE_SESSION_EPOCH_BUDGET
    is_periodic_backup = completed % CONTINUE_BACKUP_EVERY_N_EPOCHS == 0

    if is_budget_stop or is_periodic_backup:
        _push_continue_backup(trainer, completed)
    if is_budget_stop:
        print(f"\nContinuation session budget reached (total completed: "
              f"{completed}/{trainer.epochs}). Stopping gracefully.")
        trainer.stop = True


t0 = time.time()
if continue_resume_from is not None:
    model_continued = YOLO(str(continue_resume_from))
    model_continued.add_callback("on_train_epoch_end", continue_relay_callback)
    results_continued = model_continued.train(resume=True)
else:
    model_continued = YOLO(BEST_PT_FROM_ORIGINAL_RUN)
    model_continued.add_callback("on_train_epoch_end", continue_relay_callback)
    results_continued = model_continued.train(name=CONTINUE_RUN_NAME, **CONTINUE_ARGS)
wall = time.time() - t0
print(f"\nThis session: {wall/60:.1f} min")

if _continue_backup_failures:
    print(f"\nWARNING: {len(_continue_backup_failures)} backup attempt(s) failed. "
          f"Failed epochs: {[e for e, _ in _continue_backup_failures]}")
else:
    print("\nAll backup attempts this session succeeded.")


Ultralytics 8.4.108 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=5, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.3, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/kaggle/working/damage-seg.yaml, degrees=10.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=20, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.8, hsv_v=0.5, imgsz=1280, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0002, lrf=0.001, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/kaggle/working/runs/cardd_seg/cardd_yolov8s_seg_finetune-2/weights/best.pt, momentum=0.93

In [16]:
m2 = model_continued.val(data=SEG_DATA_YAML, split="test", imgsz=1280)
print("Box    mAP50:", float(m2.box.map50), " mAP50-95:", float(m2.box.map))
print("Mask   mAP50:", float(m2.seg.map50), " mAP50-95:", float(m2.seg.map))
 
import pandas as pd
rows = []
for idx, ci in enumerate(m2.box.ap_class_index):
    rows.append({
        "class": CLASS_NAMES[int(ci)],
        "box_mAP50": float(m2.box.ap50[idx]),
        "mask_mAP50": float(m2.seg.ap50[idx]),
    })
pd.DataFrame(rows).sort_values("mask_mAP50", ascending=False)

Ultralytics 8.4.108 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
val: Fast image access ✅ (ping: 0.1±0.2 ms, read: 327.4±185.4 MB/s, size: 277.0 KB)
val: Scanning /kaggle/input/datasets/m4rcuseryx/vehide-segmentation-dataset/vehide_seg/labels/test... 2047 images, 140 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 2047/2047 990.5it/s 2.1s.1ss
WARNING ⚠️ val: Cache directory /kaggle/input/datasets/m4rcuseryx/vehide-segmentation-dataset/vehide_seg/labels is not writable, cache not saved.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 128/128 1.2s/it 2:291.1ss
                   all       2047       4846      0.457      0.414      0.388      0.201      0.458      0.386      0.353      0.169
                  dent        648        825      0.416      0.264      0.239     0.0981      0.435      0.244      0.229     0.0873
               scratch       1007       2174      0.

,class,box_mAP50,mask_mAP50
4,shattered_glass,0.692946,0.660696
5,flat_tyre,0.483057,0.482821
3,broken_lamp,0.430682,0.335354
0,dent,0.239109,0.229030
2,crack,0.225331,0.209153
1,scratch,0.258147,0.203411
